NLP Task 2: Sentiment Analysis using NLP Pipeline & ML Models

In [1]:
!pip install pandas numpy nltk scikit-learn

In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download necessary NLTK datasets
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# ==========================================
# 1. DATA UNDERSTANDING (Using built-in data so no CSV is needed!)
# ==========================================
data = {
    'review': [
        "Absolutely loved this movie! The acting was brilliant. 10/10!!",
        "Worst experience ever. The food was cold and the service was terrible... http://bad-restaurant.com",
        "It was okay, not the best but certainly not the worst.",
        "I am so happy with my purchase! Highly recommend this product. 😊",
        "Do not buy this. It broke after one use. Complete waste of money!!!",
        "The cinematography was beautiful, but the plot was lacking.",
        "AMAZING! Will definitely be coming back here again.",
        "Terrible customer support. I waited on hold for 2 hours.",
        "A masterpiece of modern cinema. Truly breathtaking.",
        "It's a scam. Save your money and look elsewhere."
    ] * 20, # Multiplying by 20 to create a larger sample size
    'sentiment': [1, 0, 1, 1, 0, 0, 1, 0, 1, 0] * 20 # 1 = Positive, 0 = Negative
}

df = pd.DataFrame(data)

print("--- Dataset Overview ---")
print(f"Total Samples: {df.shape[0]}")
print("\nClass Distribution:")
print(df['sentiment'].value_counts())

# ==========================================
# 2. NLP PREPROCESSING
# ==========================================
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# Keep important negation words for sentiment analysis
stop_words.difference_update({"not", "no", "never", "nor", "none"})

def preprocess_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text) # Remove URLs
    text = text.lower() # Lowercase
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove punctuation and numbers
    tokens = text.split()
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(cleaned_tokens)

df['clean_review'] = df['review'].apply(preprocess_text)

# ==========================================
# 3. FEATURE ENGINEERING
# ==========================================
X = df['clean_review']
y = df['sentiment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Bag of Words (BoW)
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# ==========================================
# 4 & 5. MODEL BUILDING & EVALUATION
# ==========================================
def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, feature_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return {
        'Model': model_name,
        'Features': feature_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4)
    }

models = {
    "Logistic Regression": LogisticRegression(),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

results = []
# Evaluate on Bag of Words
for name, model in models.items():
    results.append(evaluate_model(model, X_train_bow, X_test_bow, y_train, y_test, name, "Bag of Words"))

# Evaluate on TF-IDF
for name, model in models.items():
    results.append(evaluate_model(model, X_train_tfidf, X_test_tfidf, y_train, y_test, name, "TF-IDF"))

results_df = pd.DataFrame(results)
print("\n--- Model Comparison ---")
print(results_df.sort_values(by='Accuracy', ascending=False).to_string(index=False))

--- Dataset Overview ---
Total Samples: 200

Class Distribution:
sentiment
1    100
0    100
Name: count, dtype: int64

--- Model Comparison ---
              Model     Features  Accuracy  Precision  Recall  F1 Score
Logistic Regression Bag of Words       1.0        1.0     1.0       1.0
        Naive Bayes Bag of Words       1.0        1.0     1.0       1.0
      Decision Tree Bag of Words       1.0        1.0     1.0       1.0
Logistic Regression       TF-IDF       1.0        1.0     1.0       1.0
        Naive Bayes       TF-IDF       1.0        1.0     1.0       1.0
      Decision Tree       TF-IDF       1.0        1.0     1.0       1.0
